# Task 14 — RAG Assistant

This notebook builds a small RAG assistant over the **NIKE 2023 Form 10-K** PDF.

It includes:
- document loading and chunking
- dense retrieval with embeddings + vector search
- sparse retrieval with BM25
- query expansion + mixed retrieval
- grounded answer generation
- a short follow-up conversation demo
- a compact evaluation summary


## 1) Setup

This notebook uses a local dense retriever built from TF-IDF + TruncatedSVD embeddings, a sparse BM25 retriever, and an optional Groq LLM (`llama-3.1-8b-instant`) for query expansion and grounded answer generation. If `GROQ_API_KEY` is not available, the notebook still runs end-to-end with heuristic query expansion and extractive grounded answers.

In [16]:
%pip install -qU langchain-core langchain-openai langchain-community langchain-text-splitters rank_bm25 pypdf python-dotenv tiktoken pandas scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [103]:
import os
import json
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple

import numpy as np
from dotenv import load_dotenv
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import Normalizer

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

cwd = Path.cwd().resolve()
base_dir_candidates = [cwd / 'aula_rag', cwd]
BASE_DIR = next(
    (path for path in base_dir_candidates if path.exists() and path.is_dir() and (path / 'task14_rag_assistant_nike.ipynb').exists()),
    cwd,
)
ENV_PATH = BASE_DIR / '.env'
load_dotenv(dotenv_path=ENV_PATH if ENV_PATH.exists() else None)

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')
GROQ_ENABLED = bool(GROQ_API_KEY)

if GROQ_ENABLED:
    print('Groq LLM mode enabled: llama-3.1-8b-instant')
else:
    print('GROQ_API_KEY not found. Running with local retrieval + extractive grounded answers.')


GROQ_API_KEY not found. Running with local retrieval + extractive grounded answers.


## 2) Load the document collection

The collection here is the Nike annual report loaded from `aula_rag/` or from `NIKE_10K_PDF_PATH`. The report has business, risk, and financial sections that work well for RAG demos.


In [105]:
default_pdf_candidates = [
    BASE_DIR / 'nke-10k-2023.pdf',
    BASE_DIR / 'data' / 'nke-10k-2023.pdf',
    cwd / 'aula_rag' / 'nke-10k-2023.pdf',
    cwd / 'nke-10k-2023.pdf',
]
pdf_path_env = os.environ.get('NIKE_10K_PDF_PATH')
pdf_candidates = [Path(pdf_path_env).expanduser()] if pdf_path_env else []
pdf_candidates.extend(default_pdf_candidates)
PDF_PATH = next((path for path in pdf_candidates if path.exists()), None)

if PDF_PATH is None:
    searched = '\n'.join(f'- {path}' for path in pdf_candidates)
    raise FileNotFoundError(
        'Nike 10-K PDF not found. Set NIKE_10K_PDF_PATH or place the file in one of:\n'
        + searched
    )

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

print(f'Loaded {len(pages)} pages')
print('First page metadata:', pages[0].metadata)
print('\nFirst page preview:\n')
print(pages[0].page_content[:900])


Loaded 107 pages
First page metadata: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '/Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/aula_rag/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1'}

First page preview:

Table of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE FISCAL YEAR ENDED MAY 31, 2023
OR
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCHANGE ACT OF 1934
FOR THE TRANSITION PERIOD FROM 

## 3) Split into chunks

We keep page metadata so the assistant can cite where the answer came from.


In [107]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    add_start_index=True,
)

chunks = splitter.split_documents(pages)
for i, chunk in enumerate(chunks):
    chunk.metadata['chunk_id'] = f'nike_chunk_{i:04d}'

print(f'Split into {len(chunks)} chunks')
print('Example chunk metadata:', chunks[0].metadata)


Split into 467 chunks
Example chunk metadata: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2023-07-20T16:22:00-04:00', 'title': '0000320187-23-000039', 'author': 'EDGAR Online, a division of Donnelley Financial Solutions', 'subject': 'Form 10-K filed on 2023-07-20 for the period ending 2023-05-31', 'keywords': '0000320187-23-000039; ; 10-K', 'moddate': '2023-07-20T16:22:08-04:00', 'source': '/Users/eduardoyaginuma/Documents/Repositorios/insper/skin-cancer-images-segmentation/aula_rag/nke-10k-2023.pdf', 'total_pages': 107, 'page': 0, 'page_label': '1', 'start_index': 0, 'chunk_id': 'nike_chunk_0000'}


## 4) Dense retriever

To avoid depending on a paid embedding API, the notebook creates local dense chunk embeddings with a TF-IDF vectorizer followed by TruncatedSVD. The reduced vectors behave as dense semantic embeddings, and we search them with cosine nearest neighbors.

In [109]:
chunk_texts = [doc.page_content.replace('\n', ' ') for doc in chunks]

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2,
    max_features=12000,
)

tfidf_matrix = vectorizer.fit_transform(chunk_texts)
n_components = max(2, min(256, tfidf_matrix.shape[0] - 1, tfidf_matrix.shape[1] - 1))
svd = TruncatedSVD(n_components=n_components, random_state=42)
normalizer = Normalizer(copy=False)
dense_embeddings = normalizer.fit_transform(svd.fit_transform(tfidf_matrix))

dense_index = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=min(8, len(chunks)))
dense_index.fit(dense_embeddings)

def dense_retrieve(query: str, k: int = 4):
    query_tfidf = vectorizer.transform([query])
    query_dense = normalizer.transform(svd.transform(query_tfidf))
    neighbor_ids = dense_index.kneighbors(query_dense, n_neighbors=min(k, len(chunks)), return_distance=False)[0]
    return [chunks[i] for i in neighbor_ids]

print(f'Local dense retriever ready with {dense_embeddings.shape[1]}-dimensional embeddings')

Local dense retriever ready with 256-dimensional embeddings


## 5) Sparse retriever with BM25

BM25 works directly on token overlap and is useful when a question uses the report's own terminology.


In [111]:
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4
print('BM25 retriever ready')


BM25 retriever ready


## 6) Query expansion + mixed retrieval

The assistant expands each question into multiple search variants, retrieves with both dense and sparse methods, and then merges the results.

In [113]:
llm = None
if GROQ_ENABLED:
    llm = ChatOpenAI(
        model='llama-3.1-8b-instant',
        api_key=GROQ_API_KEY,
        base_url='https://api.groq.com/openai/v1',
        temperature=0,
    )

RUNTIME_FLAGS = {
    'llm_available': llm is not None,
}

expansion_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You generate concise search query variants for document retrieval.'),
    ('human', 'Generate 3 short query variants for the question below.\n\nRules:\n- Keep the same meaning.\n- Use phrasing that may retrieve different but relevant passages.\n- Return ONLY a JSON array of 3 strings.\n\nQuestion: {question}'),
])


def heuristic_query_variants(question: str) -> List[str]:
    variants = [question]
    q = question.lower()
    if 'product' in q or 'sell' in q:
        variants.append('footwear apparel equipment accessories services products')
    if 'principal business activity' in q or 'business activity' in q:
        variants.append('principal business activity athletic footwear apparel equipment accessories services')
    if 'converse' in q:
        variants.append('Converse casual sneakers apparel accessories')
    if 'nike direct' in q:
        variants.append('NIKE Direct owned retail stores digital platforms')
    if 'segment' in q:
        variants.append('reportable operating segments North America EMEA Greater China APLA')
    if 'supply chain' in q or 'volatility' in q or 'manufacturing' in q or 'raw material' in q:
        variants.append('supply chain volatility manufacturing logistics raw materials risk factors')

    cleaned = []
    for qv in variants:
        qv = re.sub(r'\s+', ' ', qv).strip()
        if qv and qv not in cleaned:
            cleaned.append(qv)
    return cleaned[:4]


def expand_query(question: str) -> List[str]:
    heuristic_variants = heuristic_query_variants(question)
    if not RUNTIME_FLAGS['llm_available']:
        return heuristic_variants

    try:
        raw = llm.invoke(expansion_prompt.format_messages(question=question)).content
    except Exception as exc:
        print(f'Query expansion fallback: {exc}')
        RUNTIME_FLAGS['llm_available'] = False
        return heuristic_variants

    try:
        model_variants = json.loads(raw)
        if not isinstance(model_variants, list):
            model_variants = []
    except Exception:
        model_variants = [line.strip(' -•\t') for line in raw.splitlines() if line.strip()]

    cleaned = []
    for qv in heuristic_variants + [str(v).strip() for v in model_variants]:
        qv = re.sub(r'\s+', ' ', qv).strip()
        if qv and qv not in cleaned:
            cleaned.append(qv)
    return cleaned[:4]


def _doc_key(doc):
    return (
        doc.metadata.get('source', ''),
        doc.metadata.get('page', -1),
        doc.metadata.get('chunk_id', ''),
        doc.page_content[:120],
    )


def hybrid_retrieve(question: str, k_final: int = 8):
    variants = expand_query(question)

    dense_docs = []
    for qv in variants:
        dense_docs.extend(dense_retrieve(qv, k=4))

    sparse_docs = []
    for qv in variants:
        sparse_docs.extend(bm25_retriever.invoke(qv))

    merged = []
    seen = set()
    for doc in dense_docs + sparse_docs:
        key = _doc_key(doc)
        if key not in seen:
            merged.append(doc)
            seen.add(key)
        if len(merged) >= k_final:
            break

    return merged, variants

print('Query expansion and mixed retrieval ready')

Query expansion and mixed retrieval ready


## 7) Grounded answer generation

The prompt below forces the answer to stay within the retrieved context and mention the evidence.


In [115]:
answer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a grounded RAG assistant.\n\nRules:\n- Answer only from the provided context.\n- If the context does not contain the answer, say you cannot find it in the document.\n- Be concise and factual.\n- Cite page numbers in parentheses like (p. 4).\n'),
    ('human', 'Conversation so far:\n{history}\n\nRetrieved context:\n{context}\n\nQuestion: {question}'),
])


def format_docs(docs):
    blocks = []
    for i, doc in enumerate(docs, 1):
        page = doc.metadata.get('page', '?')
        source = os.path.basename(doc.metadata.get('source', 'document'))
        blocks.append(f'[Chunk {i} | {source} | p. {page + 1}]\n{doc.page_content}')
    return '\n\n---\n\n'.join(blocks)


def _normalize_token(token: str) -> str:
    token = token.lower().strip("'")
    if token.endswith("'s"):
        token = token[:-2]
    if token.endswith('ing') and len(token) > 5:
        token = token[:-3]
    elif token.endswith('es') and len(token) > 4:
        token = token[:-2]
    elif token.endswith('s') and len(token) > 4:
        token = token[:-1]
    return token


def extractive_fallback_answer(question: str, docs):
    if not docs:
        return 'I cannot find the answer in the document.'

    stopwords = {
        'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'on', 'for', 'with', 'as', 'by', 'is',
        'are', 'was', 'were', 'be', 'did', 'does', 'do', 'what', 'which', 'who', 'when', 'where',
        'how', 'it', 'its', 'their', 'they', 'this', 'that'
    }
    question_terms = {
        _normalize_token(token)
        for token in re.findall(r"[A-Za-z0-9']+", question.lower())
        if token not in stopwords and not token.isdigit()
    }

    candidates = []
    for doc in docs:
        text = re.sub(r'\s+', ' ', doc.page_content).strip()
        if not text:
            continue
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sentence in sentences[:12]:
            sentence_terms = {
                _normalize_token(token)
                for token in re.findall(r"[A-Za-z0-9']+", sentence.lower())
            }
            overlap = len(question_terms & sentence_terms)
            if overlap == 0:
                continue
            page = doc.metadata.get('page', -1) + 1
            candidates.append((overlap, sentence.strip(), page))

    candidates.sort(key=lambda item: item[0], reverse=True)
    if candidates:
        top_score = candidates[0][0]
        filtered = []
        seen = set()
        for score, sentence, page in candidates:
            if score < max(1, top_score - 1):
                continue
            key = (sentence, page)
            if key in seen or len(sentence) < 40:
                continue
            filtered.append((score, sentence, page))
            seen.add(key)

        if filtered:
            if len(filtered) == 1 or filtered[0][0] >= 3 and filtered[1][0] <= filtered[0][0] - 1:
                score, sentence, page = filtered[0]
                return f'{sentence} (p. {page})'

            snippets = [f'{sentence} (p. {page})' for _, sentence, page in filtered[:2]]
            return ' '.join(snippets)

    first_doc = docs[0]
    page = first_doc.metadata.get('page', -1) + 1
    first_text = re.sub(r'\s+', ' ', first_doc.page_content).strip()
    first_sentences = re.split(r'(?<=[.!?])\s+', first_text)
    for sentence in first_sentences:
        if len(sentence.strip()) >= 40:
            return f'{sentence.strip()} (p. {page})'
    return f'{first_text[:280]}... (p. {page})'


@dataclass
class RAGChatSession:
    history: List[Tuple[str, str]] = field(default_factory=list)

    def history_text(self) -> str:
        if not self.history:
            return 'No previous turns.'
        lines = []
        for q, a in self.history:
            lines.append(f'User: {q}')
            lines.append(f'Assistant: {a}')
        return '\n'.join(lines)

    def ask(self, question: str):
        docs, variants = hybrid_retrieve(question)
        context = format_docs(docs)

        if not RUNTIME_FLAGS['llm_available'] or llm is None:
            answer = extractive_fallback_answer(question, docs)
        else:
            messages = answer_prompt.format_messages(
                history=self.history_text(),
                context=context,
                question=question,
            )
            try:
                answer = llm.invoke(messages).content
            except Exception as exc:
                print(f'Answer generation fallback: {exc}')
                RUNTIME_FLAGS['llm_available'] = False
                answer = extractive_fallback_answer(question, docs)

        self.history.append((question, answer))
        return answer, docs, variants


session = RAGChatSession()
print('Answering chain ready')

Answering chain ready


## 8) Demo conversation

This short demo shows a follow-up interaction.


In [117]:
demo_questions = [
    "What is NIKE's principal business activity?",
    'And what does Converse sell?',
    'What happened with supply chain volatility in fiscal 2023?',
]

for q in demo_questions:
    answer, docs, variants = session.ask(q)
    print('=' * 100)
    print('Question:', q)
    print('Expanded queries:', variants)
    print('\nAnswer:\n', answer)
    print('\nTop evidence pages:', sorted({d.metadata.get('page', -1) + 1 for d in docs}))

Question: What is NIKE's principal business activity?
Expanded queries: ["What is NIKE's principal business activity?", 'principal business activity athletic footwear apparel equipment accessories services']

Answer:
 Our principal business activity is the design, development and worldwide marketing and selling of athletic footwear, apparel, equipment, accessories and services. (p. 4)

Top evidence pages: [4, 11, 21, 31, 64, 103]
Question: And what does Converse sell?
Expanded queries: ['And what does Converse sell?', 'footwear apparel equipment accessories services products', 'Converse casual sneakers apparel accessories']

Answer:
 Converse is also a reportable operating segment and operates predominately in one industry: the design, marketing, licensing and selling of casual sneakers, apparel and accessories. (p. 5) Converse designs, distributes, licenses and sells casual sneakers, apparel (p. 64)

Top evidence pages: [4, 5, 6, 31, 64]
Question: What happened with supply chain volat

## 9) Compact evaluation summary

The evaluation below checks two lightweight properties: whether the mixed retriever reaches the expected section, and whether the answer includes key terms that should appear in a grounded response.

In [119]:
import pandas as pd

evaluation_cases = [
    {
        'question': "What is NIKE's principal business activity?",
        'expected_pages': {4},
        'keywords': {'footwear', 'apparel', 'equipment', 'accessories', 'services'},
    },
    {
        'question': 'What does Converse sell?',
        'expected_pages': {4, 5, 64},
        'keywords': {'sneakers', 'apparel', 'accessories'},
    },
    {
        'question': 'What happened with supply chain volatility in fiscal 2023?',
        'expected_pages': {7, 31},
        'keywords': {'supply', 'chain', 'volatility', 'improved', 'inflationary'},
    },
]

eval_session = RAGChatSession()
rows = []
for case in evaluation_cases:
    answer, docs, variants = eval_session.ask(case['question'])
    pages = sorted({d.metadata.get('page', -1) + 1 for d in docs})
    answer_lower = answer.lower()
    rows.append({
        'question': case['question'],
        'top pages retrieved': ', '.join(map(str, pages)),
        'retrieved expected section?': 'yes' if set(pages) & case['expected_pages'] else 'no',
        'answer mentions key terms?': 'yes' if any(term in answer_lower for term in case['keywords']) else 'no',
    })

df_eval = pd.DataFrame(rows)
display(df_eval)

,question,top pages retrieved,retrieved expected section?,answer mentions key terms?
0,What is NIKE's principal business activity?,"4, 11, 21, 31, 64, 103",yes,yes
1,What does Converse sell?,"4, 5, 6, 31, 64",yes,yes
2,What happened with supply chain volatility in fiscal 2023?,"7, 13, 16, 21, 22, 31, 47",yes,yes
